In [11]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import geopandas as gpd
import cartopy.crs as ccrs
import cmocean  # Ensure this library is installed
from datetime import datetime

In [12]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import pandas as pd
# Define the color map
color_values = [
    [147,0,108], [111,0,144], [72,0,183], [33,0,222], [0,10,255], [0,74,255],
    [0,144,255], [0,213,255], [0,255,215], [0,255,119], [0,255,15], [96,255,0],
    [200,255,0], [255,235,0], [255,183,0], [255,131,0], [255,79,0], [255,31,0],
    [230,0,0], [165,0,0], [105,0,0]
]
sample_values = [0.0105927390978, 0.01511153517746066, 0.02218493234759939,
                 0.03259381777070279, 0.047886418593687446, 0.06831451603621097,
                 0.10029112852033921, 0.1473464383750438, 0.2164794954661175,
                 0.30882852380362497, 0.4533847704508146, 0.6661070836975002,
                 0.9786359751581525, 1.3961169989723918, 2.049610500045872,
                 3.011261431529367, 4.424106633340013, 6.311407543621453,
                 9.265646919990738, 13.612969492356, 20.0]
cmap = colors.ListedColormap(np.array(color_values)/255.0)
norm = colors.BoundaryNorm(sample_values, len(sample_values))

In [15]:
# Input and output directories
import re
input_dir = "d:/Sentinel2_WQ/process_c2rcc"  # Path to the directory containing .nc files
output_dir = "d:/Sentinel2_WQ/Final-product/CHL"  # Path to save the masked .png files
os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist
# Load and reproject the mask shapefile
#mask = gpd.read_file("d:/WAMSI/R_mask/R_mask.shp").to_crs(epsg=4326)

# Loop through all .nc files in the input directory
for file_name in os.listdir(input_dir):
    if file_name.endswith(".nc"):  # Process only .nc files
        file_path = os.path.join(input_dir, file_name)
        
        # Open the .nc file
        ds = xr.open_dataset(file_path)
        
        # Extract chlorophyll data, latitude, and longitude
        chl_data = ds["conc_chl"].where(ds["conc_chl"] != 0, np.nan).values
        lat = ds["lat"].values
        lon = ds["lon"].values
        
        # Extract start_date from attributes
        match = re.search(r"\d{8}", file_name)
        if match:
            try:
                parsed_date = datetime.strptime(match.group(), "%Y%m%d")
                formatted_date = parsed_date.strftime("%Y%m%d")
            except ValueError:
                formatted_date = "Unknown_Date"
        else:
            formatted_date = "Unknown_Date"
        
        # Create a figure with Cartopy projection
        fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
        
        # Set colormap and NaN color
        cmap_chl = cmap
        cmap_chl.set_bad(color="black")
        
        # Set extent
        extent = [lon.min(), lon.max(), lat.min(), lat.max()]
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        
        # Plot chlorophyll data
        img = ax.imshow(
            chl_data,
            extent=extent,
            cmap=cmap_chl,
            transform=ccrs.PlateCarree(),
            origin="upper",
            norm=norm
        )
        ax.set_axis_off() # for leaflet, uncomment this line to remove axis
        # for leaflet, comment the next lines
        # # Add colorbar
        # cbar = plt.colorbar(img, ax=ax, orientation='vertical', label="Chlorophyll-a (mg/m$^3$)")
        # tick_locs = [0.011, 0.07, 0.46, 1.39, 3.03, 6.01, 9, 20]
        # tick_labels = [f"{loc:.2f}" for loc in tick_locs]
        # cbar.set_ticks(tick_locs)
        # cbar.set_ticklabels(tick_labels)
        # # Add gridlines
        # gridlines = ax.gridlines(draw_labels=True, color="gray", linestyle="--", linewidth=0.5)
        # gridlines.top_labels = False
        # gridlines.right_labels = False
        
        # #Add title
        # ax.set_title(f"Chlorophyll-a Date: {formatted_date}", fontsize=14)
        # ax.set_xlabel("Longitude")
        # ax.set_ylabel("Latitude")
        # # for leaflet, comment upto this line
        # Save the plot as a .png file
        output_file_name = f"{formatted_date}.png"
        output_file_path = os.path.join(output_dir, output_file_name)
        plt.savefig(output_file_path, dpi=300, transparent=True, pad_inches=0, bbox_inches="tight")
        plt.close()  # Close the plot to free memory
        
        print(f"Saved: {output_file_path}")

Saved: d:/Sentinel2_WQ/Final-product/CHL\20250515.png
Saved: d:/Sentinel2_WQ/Final-product/CHL\20250520.png
Saved: d:/Sentinel2_WQ/Final-product/CHL\20250527.png


In [ ]:
# Input and output directories
input_dir = "d:/Sentinel2_WQ/process_c2rcc"  # Path to the directory containing .nc files
output_dir = "d:/Sentinel2_WQ/Final-product/TSM"  # Path to save the masked .png files
os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist

# Loop through all .nc files in the input directory
for file_name in os.listdir(input_dir):
    if file_name.endswith(".nc"):  # Process only .nc files
        file_path = os.path.join(input_dir, file_name)
        
        # Open the .nc file
        ds = xr.open_dataset(file_path)
        
        
        # Try to extract total suspended matter (TSM) data
        try:
            tsm_data = ds["conc_tsm"].where(ds["conc_tsm"] != 0, np.nan).values
        except KeyError:
            print(f"Variable 'conc_tsm' not found in {file_name}. Skipping this file.")
            continue
        
        # Extract latitude and longitude ranges
        lat = ds["lat"].values
        lon = ds["lon"].values
        
        # Extract start_date from attributes
        # Extract start_date from attributes
        match = re.search(r"\d{8}", file_name)
        if match:
            try:
                parsed_date = datetime.strptime(match.group(), "%Y%m%d")
                formatted_date = parsed_date.strftime("%Y%m%d")
            except ValueError:
                formatted_date = "Unknown_Date"
        else:
            formatted_date = "Unknown_Date"
        
        # Replace NaN values with a specific color (black)
        cmap = cmocean.cm.turbid  # Using cmocean's matter colormap
        cmap.set_bad(color="black")
        
        # Plot the data
        fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
        img=ax.imshow(
            tsm_data,
            cmap=cmap,
            vmin=0,  # Set minimum value for linear scale
            vmax=4,  # Set maximum value for linear scale
            origin="upper",
            extent=[lon.min(), lon.max(), lat.min(), lat.max()]
        )
        ax.set_axis_off()
        # Save the plot as a .png file
        output_file_name = f"{formatted_date}.png"
        output_file_path = os.path.join(output_dir, output_file_name)
        plt.savefig(output_file_path, dpi=300, transparent=True, pad_inches=0, bbox_inches="tight")
        plt.close()  # Close the plot to free memory
        
        print(f"Saved: {output_file_path}")

Saved: d:/Sentinel2_WQ/Final-product/TSM\20250515.png
Saved: d:/Sentinel2_WQ/Final-product/TSM\20250520.png
Saved: d:/Sentinel2_WQ/Final-product/TSM\20250527.png


In [10]:
# Input and output directories
input_dir = "d:/Sentinel2_WQ/process_cdom"  # Path to the directory containing .nc files
output_dir = "d:/Sentinel2_WQ/Final-product/CDOM"  # Path to save the masked .png files
os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist

# Loop through all .nc files in the input directory
for file_name in os.listdir(input_dir):
    if file_name.endswith(".nc"):  # Process only .nc files
        file_path = os.path.join(input_dir, file_name)
        
        # Open the .nc file
        ds = xr.open_dataset(file_path)
        
        # Extract chlorophyll data, latitude, and longitude
        cdom_data = ds["CDOM"].where(ds["CDOM"] != 0, np.nan).values
        lat = ds["lat"].values
        lon = ds["lon"].values
        
        # Extract start_date from attributes
        match = re.search(r"\d{8}", file_name)
        if match:
            try:
                parsed_date = datetime.strptime(match.group(), "%Y%m%d")
                formatted_date = parsed_date.strftime("%Y%m%d")
            except ValueError:
                formatted_date = "Unknown_Date"
        else:
            formatted_date = "Unknown_Date"
        
        # Create a figure with Cartopy projection
        fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
        # Set colormap and NaN color
        cmap_cdom = plt.cm.YlOrBr
        cmap_cdom.set_bad(color="black")
        
        # Set extent
        img = ax.imshow(
            cdom_data,
            extent=extent,
            cmap=cmap_cdom,
            transform=ccrs.PlateCarree(),
            origin="upper",
              vmin=0, vmax=4
        )
        ax.set_axis_off()
      # Save the plot as a .png file
        output_file_name = f"{formatted_date}.png"
        output_file_path = os.path.join(output_dir, output_file_name)
        plt.savefig(output_file_path, dpi=300, transparent=True, pad_inches=0, bbox_inches="tight")
        plt.close()  # Close the plot to free memory
        
        print(f"Saved: {output_file_path}")

Saved: d:/Sentinel2_WQ/Final-product/CDOM\20250515.png
Saved: d:/Sentinel2_WQ/Final-product/CDOM\20250520.png
Saved: d:/Sentinel2_WQ/Final-product/CDOM\20250527.png
